# Verify hardware

In [ ]:
!nvidia-smi

Thu Nov 20 12:50:02 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   29C    P8              4W /  160W |     494MiB /   8188MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
Note: you may need to restart the kernel to use updated packages.


# Test PyTorch GPU access

In [3]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


CUDA Available: True
GPU: NVIDIA GeForce RTX 4060 Ti


# SPO Extraction and SPO Embeddings

## SPO extraction helper

In [4]:
pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------------------------- ------- 10.5/12.8 MB 54.4 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 57.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [5]:
# build_spo_and_spo_embeddings.py
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import spacy
from transformers import AutoTokenizer, AutoModel
import torch

DATA_DIR = "D:/multimodal_pipeline/data2"
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test_public.tsv"),
}

SPO_EMB_DIR = os.path.join(DATA_DIR, "spo_embeddings")  # will contain train/, validate/, test/
os.makedirs(SPO_EMB_DIR, exist_ok=True)
for split in ["train", "validate", "test"]:
    os.makedirs(os.path.join(SPO_EMB_DIR, split), exist_ok=True)

SPO_BERT_MODEL = "bert-base-uncased"
MAX_SPO_SEQ_LEN = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

print(f"Loading BERT model for SPO embeddings: {SPO_BERT_MODEL}")
spo_tokenizer = AutoTokenizer.from_pretrained(SPO_BERT_MODEL)
spo_bert = AutoModel.from_pretrained(SPO_BERT_MODEL).to(device)
spo_bert.eval()


c:\Users\NWU\miniconda3\envs\multimodal_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading spaCy model...
Loading BERT model for SPO embeddings: bert-base-uncased


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

## SPO extractor (very simple dependency-based heuristic)

In [6]:
def extract_spo_triples(text, max_triples=3):
    """
    Basic SPO extractor using spaCy dependencies.
    Returns list of (subject, predicate, object) tuples.
    """
    doc = nlp(text)
    triples = []

    for sent in doc.sents:
        subj = None
        obj = None
        verb = None

        for token in sent:
            # subject
            if "subj" in token.dep_:
                subj = token.text

            # object
            if "obj" in token.dep_:
                obj = token.text

            # verb (root or main verb)
            if token.pos_ == "VERB" and (token.dep_ == "ROOT" or token.head == token):
                verb = token.lemma_

        if subj and verb and obj:
            triples.append((subj, verb, obj))
            if len(triples) >= max_triples:
                break

    return triples


### Encode SPO triples into a single vector per post

In [7]:
@torch.no_grad()
def encode_spo_triples(triples):
    """
    Encode a list of SPO triples as a single BERT [CLS] embedding (768-d).
    If no triples: return zeros.
    """
    if not triples:
        return np.zeros(768, dtype=np.float32)

    texts = [f"{s} [SEP] {p} [SEP] {o}" for (s, p, o) in triples]
    joined = " [SEP] ".join(texts)

    enc = spo_tokenizer(
        joined,
        truncation=True,
        padding="max_length",
        max_length=MAX_SPO_SEQ_LEN,
        return_tensors="pt"
    ).to(device)

    outputs = spo_bert(**enc)
    cls_emb = outputs.last_hidden_state[:, 0, :]  # (1, 768)
    return cls_emb.squeeze(0).cpu().numpy().astype(np.float32)


# Main SPO pipeline

In [8]:
def process_split(split):
    tsv_path = TSV_FILES[split]
    out_dir = os.path.join(SPO_EMB_DIR, split)
    print(f"\n=== Processing SPO for {split} from {tsv_path} ===")

    df = pd.read_csv(tsv_path, sep="\t")
    print(f"Loaded {len(df)} rows")

    spo_meta_records = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        post_id = str(row["id"])
        text = str(row.get("clean_title", row.get("title", "")))

        emb_path = os.path.join(out_dir, f"{post_id}.npy")
        if os.path.exists(emb_path):
            spo_meta_records.append({"id": post_id, "spo_emb_path": emb_path})
            continue

        triples = extract_spo_triples(text)
        emb = encode_spo_triples(triples)
        np.save(emb_path, emb)

        spo_meta_records.append({"id": post_id, "spo_emb_path": emb_path})

    meta_df = pd.DataFrame(spo_meta_records)
    meta_csv_path = os.path.join(SPO_EMB_DIR, f"{split}_spo_metadata.csv")
    meta_df.to_csv(meta_csv_path, index=False)
    print(f"SPO metadata saved to {meta_csv_path}")


#### Run for all splits

In [9]:
if __name__ == "__main__":
    for split in ["train", "validate", "test"]:
        process_split(split)



=== Processing SPO for train from D:/multimodal_pipeline/data2\multimodal_train.tsv ===
Loaded 564000 rows


100%|██████████| 564000/564000 [2:50:08<00:00, 55.25it/s]   


SPO metadata saved to D:/multimodal_pipeline/data2\spo_embeddings\train_spo_metadata.csv

=== Processing SPO for validate from D:/multimodal_pipeline/data2\multimodal_validate.tsv ===
Loaded 59342 rows


100%|██████████| 59342/59342 [10:30<00:00, 94.09it/s] 


SPO metadata saved to D:/multimodal_pipeline/data2\spo_embeddings\validate_spo_metadata.csv

=== Processing SPO for test from D:/multimodal_pipeline/data2\multimodal_test_public.tsv ===
Loaded 59319 rows


100%|██████████| 59319/59319 [10:32<00:00, 93.84it/s] 


SPO metadata saved to D:/multimodal_pipeline/data2\spo_embeddings\test_spo_metadata.csv


# Graph + GNN Embeddings (GCN/GAT)

## Graph building + GNN training (sketch but runnable skeleton)

In [22]:
# build_graph_embeddings.py
import os
import json
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from transformers import AutoTokenizer, AutoModel

DATA_DIR = "D:/multimodal_pipeline/data2"
GRAPH_DIR = os.path.join(DATA_DIR, "graph_embeddings")
os.makedirs(GRAPH_DIR, exist_ok=True)

BERT_MODEL = "bert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

MAX_TEXT_LEN = 64
GRAPH_EMB_DIM = 256


#### Encode text to BERT node features

In [23]:
@torch.no_grad()
def encode_text(text: str) -> np.ndarray:
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_TEXT_LEN,
        return_tensors="pt"
    ).to(device)

    out = bert_model(**enc)
    cls = out.last_hidden_state[:, 0, :]  # (1, 768)
    return cls.squeeze(0).cpu().numpy().astype(np.float32)


#### Build a simple graph for one split

In [24]:
def build_graph_for_split(split):
    tsv_path = os.path.join(DATA_DIR, f"multimodal_{split}.tsv")
    df = pd.read_csv(tsv_path, sep="\t")
    print(f"\n=== Building graph for {split} ({len(df)} posts) ===")

    ids = df["id"].astype(str).tolist()
    id2idx = {pid: i for i, pid in enumerate(ids)}

    # Node features: BERT CLS
    x = np.zeros((len(df), 768), dtype=np.float32)
    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
        text = str(row.get("clean_title", row.get("title", "")))
        x[i] = encode_text(text)

    # Build edges – simple heuristics
    edges = []
    # group by subreddit
    if "subreddit" in df.columns:
        for _, sub_df in df.groupby("subreddit"):
            idxs = sub_df.index.tolist()
            for i in range(len(idxs) - 1):
                u = idxs[i]
                v = idxs[i + 1]
                edges.append((u, v))
                edges.append((v, u))

    # group by domain
    if "domain" in df.columns:
        for _, ddf in df.groupby("domain"):
            idxs = ddf.index.tolist()
            for i in range(len(idxs) - 1):
                u = idxs[i]
                v = idxs[i + 1]
                edges.append((u, v))
                edges.append((v, u))

    if not edges:
        raise RuntimeError("No edges constructed – adjust heuristics")

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    data = Data(
        x=torch.from_numpy(x),
        edge_index=edge_index
    )

    # Save mapping
    os.makedirs(os.path.join(GRAPH_DIR, split), exist_ok=True)
    with open(os.path.join(GRAPH_DIR, f"{split}_id2idx.json"), "w") as f:
        json.dump(id2idx, f)

    return data, df, id2idx


#### Define a small GAT/GCN

In [25]:
import torch
import torch.nn as nn
from torch_scatter import scatter_mean

class GraphEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, out_dim=128):
        super().__init__()

        self.lin1 = nn.Linear(input_dim, hidden_dim)
        self.lin2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        # Simple mean aggregation
        row, col = edge_index
        agg = scatter_mean(x[col], row, dim=0, dim_size=x.size(0))

        h = torch.relu(self.lin1(agg))
        h = self.lin2(h)
        return h


#### Train unsupervised (simple smoothing) and save node embeddings

In [26]:
GRAPH_DEVICE = torch.device("cpu")   #FORCE CPU


def train_graph_encoder(data: Data, epochs=5):
    """
    Trains a lightweight graph encoder on CPU to avoid CUDA OOM.
    Returns node embeddings as numpy (float32).
    """

    # Move graph data to CPU
    x = data.x.to(GRAPH_DEVICE)
    edge_index = data.edge_index.to(GRAPH_DEVICE)

    # Initialize CPU graph encoder
    model = GraphEncoder(
        input_dim =x.size(1),
        hidden_dim=128,     # small & safe
        out_dim=128
    ).to(GRAPH_DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    model.train()
    for epoch in range(1, epochs + 1):
        opt.zero_grad()

        out = model(x, edge_index)

        # Smoothness reconstruction loss
        target = x[:, :out.size(1)]
        loss = ((out - target) ** 2).mean()

        loss.backward()
        opt.step()

        print(f"[GNN] Epoch {epoch}/{epochs} - Loss: {loss.item():.4f}")

    # Final CPU embeddings
    model.eval()
    with torch.no_grad():
        emb = model(x, edge_index).cpu().numpy().astype(np.float32)

    # CPU cleanup only
    del model, x, edge_index, out, target

    return emb


## Main

In [27]:
def process_split(split):
    data, df, id2idx = build_graph_for_split(split)
    emb = train_graph_encoder(data, epochs=5)

    split_dir = os.path.join(GRAPH_DIR, split)
    records = []
    for post_id, idx in id2idx.items():
        path = os.path.join(split_dir, f"{post_id}.npy")
        np.save(path, emb[idx])
        records.append({"id": post_id, "graph_emb_path": path})

    meta_df = pd.DataFrame(records)
    meta_df.to_csv(os.path.join(GRAPH_DIR, f"{split}_graph_metadata.csv"), index=False)
    print(f"Graph embeddings saved for {split}")

if __name__ == "__main__":
    for split in ["train", "validate", "test"]:
        process_split(split)



=== Building graph for train (564000 posts) ===


100%|██████████| 564000/564000 [1:08:57<00:00, 136.30it/s]


[GNN] Epoch 1/5 - Loss: 0.1882
[GNN] Epoch 2/5 - Loss: 0.1583
[GNN] Epoch 3/5 - Loss: 0.1380
[GNN] Epoch 4/5 - Loss: 0.1231
[GNN] Epoch 5/5 - Loss: 0.1105
Graph embeddings saved for train

=== Building graph for validate (59342 posts) ===


100%|██████████| 59342/59342 [13:42<00:00, 72.12it/s] 


[GNN] Epoch 1/5 - Loss: 0.2030
[GNN] Epoch 2/5 - Loss: 0.1711
[GNN] Epoch 3/5 - Loss: 0.1523
[GNN] Epoch 4/5 - Loss: 0.1393
[GNN] Epoch 5/5 - Loss: 0.1285
Graph embeddings saved for validate

=== Building graph for test (59319 posts) ===


100%|██████████| 59319/59319 [14:07<00:00, 69.98it/s] 


[GNN] Epoch 1/5 - Loss: 0.2008
[GNN] Epoch 2/5 - Loss: 0.1680
[GNN] Epoch 3/5 - Loss: 0.1450
[GNN] Epoch 4/5 - Loss: 0.1272
[GNN] Epoch 5/5 - Loss: 0.1133
Graph embeddings saved for test


# Full multimodal GPU training

# Config

In [28]:
pip install matplotlib

In [27]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
import time
import matplotlib.pyplot as plt


## Dataset

In [28]:
DATA_DIR = "D:/multimodal_pipeline/data2"
IMAGE_DIRS = {
    "train": os.path.join(DATA_DIR, "train_images"),
    "validate": os.path.join(DATA_DIR, "validate_images"),
    "test": os.path.join(DATA_DIR, "test_images"),
}
TSV_FILES = {
    "train": os.path.join(DATA_DIR, "multimodal_train.tsv"),
    "validate": os.path.join(DATA_DIR, "multimodal_validate.tsv"),
    "test": os.path.join(DATA_DIR, "multimodal_test.tsv"),
}
SPO_EMB_DIR = os.path.join(DATA_DIR, "spo_embeddings")
GRAPH_EMB_DIR = os.path.join(DATA_DIR, "graph_embeddings")

TEXT_MODEL_NAME = "bert-base-uncased"
MAX_SEQ_LEN = 64
NUM_CLASSES = 6  

BATCH_SIZE = 16
NUM_EPOCHS = 15 
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 1e-4
PATIENCE = 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


## Transforms

In [29]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


## Tokenizer / text encoder for on-the-fly text embeddings

In [30]:
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_bert = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(device)
text_bert.eval()


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

# Dataset class

In [31]:
class MultimodalFakedditDataset(Dataset):
    def __init__(self, split, label_column="6_way_label", transform=None):
        self.split = split
        self.tsv_path = TSV_FILES[split]
        self.image_root = IMAGE_DIRS[split]
        self.spo_dir = os.path.join(SPO_EMB_DIR, split)
        self.graph_dir = os.path.join(GRAPH_EMB_DIR, split)
        self.label_column = label_column
        self.transform = transform

        self.df = pd.read_csv(self.tsv_path, sep="\t")
        print(f"[{split}] Loaded {len(self.df)} rows from {self.tsv_path}")

    def __len__(self):
        return len(self.df)

    def _load_image(self, post_id):
        img_path = os.path.join(self.image_root, f"{post_id}.jpg")
        if not os.path.exists(img_path):
            # fallback: black image
            img = Image.new("RGB", (224, 224), (0, 0, 0))
        else:
            img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return img

    def _load_spo_emb(self, post_id):
        path = os.path.join(self.spo_dir, f"{post_id}.npy")
        if os.path.exists(path):
            arr = np.load(path).astype(np.float32)
        else:
            arr = np.zeros(768, dtype=np.float32)  # BERT hidden size
        return torch.from_numpy(arr)

    def _load_graph_emb(self, post_id):
        path = os.path.join(self.graph_dir, f"{post_id}.npy")
        if os.path.exists(path):
            arr = np.load(path).astype(np.float32)
        else:
            arr = np.zeros(256, dtype=np.float32)
        return torch.from_numpy(arr)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        post_id = str(row["id"])
        text = str(row.get("clean_title", row.get("title", "")))
        label = int(row[self.label_column])

        # image
        image_tensor = self._load_image(post_id)

        # text: tokenize here, encode in model (to use AMP + GPU better)
        enc = text_tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_SEQ_LEN,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)      # (seq_len,)
        attention_mask = enc["attention_mask"].squeeze(0)

        # SPO and graph precomputed
        spo_emb = self._load_spo_emb(post_id)       # (768,)
        graph_emb = self._load_graph_emb(post_id)   # (256,)

        return input_ids, attention_mask, image_tensor, spo_emb, graph_emb, label


# Data loaders

In [32]:
train_dataset = MultimodalFakedditDataset("train", transform=train_transform)
val_dataset = MultimodalFakedditDataset("validate", transform=val_test_transform)
test_dataset = MultimodalFakedditDataset("test", transform=val_test_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True if device.type == "cuda" else False
)


[train] Loaded 564000 rows from D:/multimodal_pipeline/data2\multimodal_train.tsv
[validate] Loaded 59342 rows from D:/multimodal_pipeline/data2\multimodal_validate.tsv
[test] Loaded 59319 rows from D:/multimodal_pipeline/data2\multimodal_test.tsv


## Multimodal Model (with hybrid + adaptive fusion)

In [33]:
class MultimodalFakeNewsModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        # ---- Text encoder (BERT) ----
        self.text_bert = text_bert  # reuse global to avoid re-loading
        self.text_hidden = self.text_bert.config.hidden_size  # 768
        self.text_proj = nn.Linear(self.text_hidden, 256)

        # ---- Image encoder (ResNet50) ----
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        for p in resnet.parameters():
            p.requires_grad = False
        img_feat_dim = resnet.fc.in_features  # 2048
        resnet.fc = nn.Identity()
        self.image_encoder = resnet
        self.image_proj = nn.Linear(img_feat_dim, 256)

        # ---- SPO encoder ----
        self.spo_in_dim = 768
        self.spo_proj = nn.Sequential(
            nn.Linear(self.spo_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ---- Graph encoder ----
        self.graph_in_dim = 128
        self.graph_proj = nn.Sequential(
            nn.Linear(self.graph_in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # ---- Adaptive gates (one per modality) ----
        self.gate_text = nn.Linear(256, 1)
        self.gate_image = nn.Linear(256, 1)
        self.gate_spo = nn.Linear(256, 1)
        self.gate_graph = nn.Linear(256, 1)

        # ---- Fusion MLP (Hybrid late fusion) ----
        fused_dim = 256 * 4
        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        # Final classifier
        self.classifier = nn.Linear(256, num_classes)

    def encode_text(self, input_ids, attention_mask):
        out = self.text_bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]        # (B, 768)
        return self.text_proj(cls)                  # (B, 256)

    def encode_image(self, images):
        feats = self.image_encoder(images)          # (B, 2048)
        return self.image_proj(feats)               # (B, 256)

    def encode_spo(self, spo_embs):
        return self.spo_proj(spo_embs)              # (B, 256)

    def encode_graph(self, graph_embs):
        return self.graph_proj(graph_embs)          # (B, 256)

    def forward(self, input_ids, attention_mask, images, spo_embs, graph_embs):
        # Encode modalities
        text_f = self.encode_text(input_ids, attention_mask)
        img_f = self.encode_image(images)
        spo_f = self.encode_spo(spo_embs)
        graph_f = self.encode_graph(graph_embs)

        # Adaptive gates
        t_gate = torch.sigmoid(self.gate_text(text_f))   # (B, 1)
        v_gate = torch.sigmoid(self.gate_image(img_f))
        s_gate = torch.sigmoid(self.gate_spo(spo_f))
        g_gate = torch.sigmoid(self.gate_graph(graph_f))

        text_f = text_f * t_gate
        img_f = img_f * v_gate
        spo_f = spo_f * s_gate
        graph_f = graph_f * g_gate

        # Fuse
        fused = torch.cat([text_f, img_f, spo_f, graph_f], dim=1)
        fused = self.fusion(fused)
        logits = self.classifier(fused)
        return logits


# Training utilities

In [34]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.best_state_dict = None
        self.should_stop = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


## Training / evaluation functions

In [35]:
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Train", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Val", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc


C:\Users\NWU\AppData\Local\Temp\ipykernel_7256\4127237080.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [36]:
batch = next(iter(train_loader))
print("Batch obtained!")


Batch obtained!


In [37]:
pip install tensorboard


Note: you may need to restart the kernel to use updated packages.


## Training loops and curves

In [40]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir="runs/multimodal_experiment")

model = MultimodalFakeNewsModel(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

early_stopper = EarlyStopping(patience=PATIENCE, min_delta=0.0)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_model_path = os.path.join(DATA_DIR, "best_multimodal_model_gpu.pth")

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    # ---- Fast TensorBoard Logging ----
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)
    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)
    writer.add_scalar("LR", optimizer.param_groups[0]['lr'], epoch)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start
    print(f"Epoch {epoch}: "
          f"Train Loss={train_loss:.4f}, Acc={train_acc:.4f} | "
          f"Val Loss={val_loss:.4f}, Acc={val_acc:.4f} | "
          f"Time={elapsed:.1f}s")

    early_stopper.step(val_loss, model)
    if early_stopper.should_stop:
        print("🚨 Early stopping triggered!")
        break

# ---- Load Best Model ----
if early_stopper.best_state_dict is not None:
    model.load_state_dict(early_stopper.best_state_dict)

torch.save(model.state_dict(), best_model_path)
print("Best multimodal model saved to:", best_model_path)



=== Epoch 1/15 ===


Train:   0%|          | 0/35250 [00:00<?, ?it/s]C:\Users\NWU\AppData\Local\Temp\ipykernel_7256\4127237080.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):


Epoch 1: Train Loss=0.4098, Acc=0.8562 | Val Loss=0.3853, Acc=0.8647 | Time=188048.6s

=== Epoch 2/15 ===


KeyboardInterrupt: 

## Plot learning curves

In [ ]:
def plot_learning_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Loss Curves")
    plt.legend()
    plt.grid(True)

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["val_acc"], label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Accuracy Curves")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_learning_curves(history)


# Evaluation

In [ ]:

# LOAD BEST MODEL FOR TESTING
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

all_labels = []
all_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing", leave=False):
        input_ids, attention_mask, images, spo_embs, graph_embs, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        images = images.to(device)
        spo_embs = spo_embs.to(device)
        graph_embs = graph_embs.to(device)
        labels = labels.to(device)

        logits = model(input_ids, attention_mask, images, spo_embs, graph_embs)
        preds = logits.argmax(dim=1)

        all_labels.extend(labels.cpu().tolist())
        all_preds.extend(preds.cpu().tolist())

# Convert to numpy
y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Classification report
target_names = [f"class_{i}" for i in range(NUM_CLASSES)]
report_dict = classification_report(y_true, y_pred, target_names=target_names, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
print("\n--- Classification Report ---")
print(report_df)

# Save report to CSV
report_csv_path = os.path.join(DATA_DIR, "multimodal_test_classification_report.csv")
report_df.to_csv(report_csv_path)
print("Classification report saved to:", report_csv_path)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix — Multimodal Test Set")
plt.tight_layout()
cm_path = os.path.join(DATA_DIR, "multimodal_test_confusion_matrix.png")
plt.savefig(cm_path)
plt.show()
print("Confusion matrix saved to:", cm_path)
